# Aula 4 — Notebook 3

## Autonomia controlada, API e empacotamento

O workflow é multiagente, mas nenhum agente recebe autorização irrestrita. A triagem escolhe ferramentas de uma lista permitida; o planejador propõe uma ação; a política determinística decide se ela pode prosseguir; e ações sensíveis são encaminhadas para revisão humana.


## 1. Níveis de autonomia

A presença de múltiplos agentes não implica maior autonomia operacional. Quantidade de agentes e poder de ação são decisões diferentes.

Neste projeto:

- os agentes podem interpretar, revisar e propor;
- as ferramentas são executadas por funções controladas;
- ações externas são apenas simuladas;
- risco médio, risco alto ou irreversibilidade exigem aprovação;
- ações proibidas são bloqueadas independentemente da recomendação do LLM.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INCIDENT_AGENT_DIR = PROJECT_ROOT / "target_project/incident_agent"
sys.path.insert(0, str(PROJECT_ROOT))

from shared.aula_4.validation import validate_action_policy

safe_action = {
    "action_name": "collect_additional_metrics",
    "risk_level": "low",
    "reversible": True,
    "parameters": {"window_minutes": 15},
    "justification": "Faltam evidências para confirmar a hipótese.",
    "requires_approval_recommendation": False,
}

validate_action_policy(safe_action)

In [ ]:
high_risk_action = {
    "action_name": "rotate_credential",
    "risk_level": "high",
    "reversible": False,
    "parameters": {"credential_id": "masked"},
    "justification": "Existe suspeita de exposição de credencial.",
    "requires_approval_recommendation": True,
}

validate_action_policy(high_risk_action)

## 2. Human-in-the-loop como interrupção intencional

Encaminhar uma execução para uma pessoa não é necessariamente uma falha. Pode ser uma decisão de projeto.

No workflow da aula, a revisão humana ocorre quando:

- a validação falha e o limite de tentativas foi atingido;
- a investigação declara incerteza relevante;
- a ação proposta é de médio ou alto risco;
- a ação não é reversível;
- a política proíbe a ação.

O projeto registra a interrupção como estado final `human_review_required`. Em uma evolução posterior, esse estado poderia ser persistido em um banco e retomado depois de uma aprovação autenticada.

## 3. CLI e API como portas de entrada

A CLI e a API não devem possuir versões independentes da regra de negócio.

Ambas chamam:

```python
run_workflow(incident, settings, knowledge_dir)
```

Essa separação permite testar o workflow sem iniciar um servidor HTTP e evita inconsistências entre interfaces.

### CLI

```bash
python -m target_project.incident_agent.app.cli investigate --incident-file target_project/incident_agent/data/incidents/incident_documented.json
```

### API

```bash
uvicorn target_project.incident_agent.app.api:app --reload
```

Em seguida, a interface interativa estará disponível em `/docs`.

In [ ]:
# Exemplo de payload enviado à API
api_payload = {
    "incident_id": "INC-2001",
    "title": "Falha em processamento",
    "description": "O job noturno falhou após timeout na origem de dados.",
    "severity": "high",
    "source": "monitoring",
}
api_payload

## 4. Empacotamento com Docker

Docker é apresentado aqui como mecanismo de reprodutibilidade local, não como sinônimo de produção.

O contêiner padroniza:

- versão do Python;
- instalação das dependências;
- diretório de trabalho;
- comando de inicialização;
- porta exposta.

Ainda permanecem questões fora do escopo desta aula, como:

- gerenciamento corporativo de segredos;
- autenticação e autorização;
- políticas de rede;
- escalabilidade;
- alta disponibilidade;
- observabilidade distribuída;
- governança e retenção de dados.

In [ ]:
dockerfile_path = PROJECT_ROOT / "target_project/incident_agent/Dockerfile"
print(dockerfile_path.read_text(encoding="utf-8"))

## 5. Teste ponta a ponta sugerido

Para concluir a formação, execute os seguintes cenários:

| Cenário | Comportamento esperado |
|---|---|
| Incidente bem documentado | investigação baseada no runbook |
| Evidências insuficientes | baixa confiança ou revisão humana |
| Saída inválida | nova tentativa limitada |
| Ação de médio risco | interrupção para aprovação |
| Ação proibida | bloqueio determinístico |
| Falha de configuração | encerramento rápido com mensagem clara |

O objetivo não é apenas verificar se o modelo gerou um texto plausível. É confirmar se o sistema respeitou seus contratos, limites e rotas.

## 6. Exercício final — escolhendo a arquitetura

Compare três alternativas para o mesmo caso:

1. um único agente com todas as responsabilidades;
2. quatro agentes especializados, como no projeto;
3. dois agentes: executor e revisor.

Não escolha pela quantidade de agentes. Defenda a alternativa usando critérios como complexidade da tarefa, necessidade de revisão independente, custo, latência, explicabilidade, volume de execução e impacto de erros.

Depois, altere o grafo removendo um agente. Verifique quais contratos, rotas e métricas precisam ser ajustados. O exercício demonstra que frameworks facilitam a orquestração, mas não eliminam decisões de arquitetura.


## Encerramento do curso

A progressão completa pode ser resumida assim:

```text
Aula 1: prompt → fluxo → agente → ferramenta
Aula 2: agente único → responsabilidades especializadas → multiagentes
Aula 3: resposta → contexto controlado → avaliação → confiabilidade
Aula 4: protótipo → aplicação organizada → rastreabilidade → autonomia controlada
```

Um sistema agêntico não está concluído quando consegue gerar uma resposta convincente. Ele precisa ser compreensível, testável, rastreável e limitado por decisões de engenharia explícitas.